In [11]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 24
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


In [1]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
import wbgapi as wb
wb.db = 2

# Check WB tariff indicators
tariff_candidates = [
    'TM.TAX.MRCH.SM.AR.ZS',  # Tariff rate, applied, simple mean
    'TM.TAX.MRCH.WM.AR.ZS',  # Tariff rate, applied, weighted mean
    'TM.TAX.MANF.SM.AR.ZS',  # Tariff rate on manufactures
]

for code in tariff_candidates:
    try:
        df = wb.data.DataFrame(code, time=range(2020, 2023), labels=False)
        print(f"IN WDI: {code} — shape {df.shape}")
    except:
        print(f"NOT IN WDI: {code}")

IN WDI: TM.TAX.MRCH.SM.AR.ZS — shape (266, 3)
IN WDI: TM.TAX.MRCH.WM.AR.ZS — shape (266, 3)
IN WDI: TM.TAX.MANF.SM.AR.ZS — shape (266, 3)


In [2]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
import wbgapi as wb
wb.db = 2

check_indicators = {
    'Carbon pricing ETS': 'EN.CLC.GHGR.MT.CE',
    'Carbon tax': 'EN.CLC.CARP.ZS',
    'WTO TFA': 'TT.TRF.FACT.XD.ZS',
}

for name, code in check_indicators.items():
    try:
        df = wb.data.DataFrame(code, time=range(2020, 2023), labels=False)
        print(f"IN WDI: {name} ({code}) — shape {df.shape}")
    except:
        print(f"NOT IN WDI: {name} ({code})")

NOT IN WDI: Carbon pricing ETS (EN.CLC.GHGR.MT.CE)
NOT IN WDI: Carbon tax (EN.CLC.CARP.ZS)
NOT IN WDI: WTO TFA (TT.TRF.FACT.XD.ZS)


In [5]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
from datetime import datetime, timedelta

IMAPP_BASE = "https://www.elibrary-areaer.imf.org/Macroprudential/Documents"

def get_latest_imapp_url():
    """Try recent dates to find the latest iMaPP ZIP file."""
    # Try dates from today backwards for up to 2 years
    check_date = datetime.today()
    for _ in range(730):
        date_str = check_date.strftime("%Y-%m-%d")
        url = f"{IMAPP_BASE}/iMaPP_database-{date_str}.zip"
        try:
            r = requests.head(url, timeout=5, allow_redirects=True)
            if r.status_code == 200 and 'zip' in r.headers.get('Content-Type', '').lower():
                print(f"Found: {url}")
                return url, date_str
        except:
            pass
        check_date -= timedelta(days=1)
    return None, None

print("Searching for latest iMaPP ZIP...")
IMAPP_URL, IMAPP_DATE = get_latest_imapp_url()
print(f"Latest: {IMAPP_DATE}")

Searching for latest iMaPP ZIP...
Found: https://www.elibrary-areaer.imf.org/Macroprudential/Documents/iMaPP_database-2025-09-29.zip
Latest: 2025-09-29


In [12]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
import re

HEADERS = BROWSER_HEADERS

# Fetch DataMapper page to find Fiscal Rules Excel URL
fr_page = requests.get(
    'https://www.imf.org/external/datamapper/fiscalrules/map/map.htm',
    headers=HEADERS,
    timeout=30
)
print(f"Status: {fr_page.status_code}")

# Search for Excel/CSV/download links
xlsx_links = re.findall(r'https?://[^\s"\'<>]+\.xlsx', fr_page.text)
ashx_links = re.findall(r'https?://[^\s"\'<>]+\.ashx', fr_page.text)
media_links = re.findall(r'/-/media/[^\s"\'<>]+', fr_page.text)

print(f"XLSX links: {xlsx_links[:5]}")
print(f"ASHX links: {ashx_links[:5]}")
print(f"Media links (fiscal/rules): {[l for l in media_links if 'fiscal' in l.lower() or 'rules' in l.lower()][:5]}")
print(f"\nFirst 1000 chars:")
print(fr_page.text[:1000])

Status: 403
XLSX links: []
ASHX links: []
Media links (fiscal/rules): []

First 1000 chars:
<HTML><HEAD>
<TITLE>Access Denied</TITLE>
</HEAD><BODY>
<H1>Access Denied</H1>
 
You don't have permission to access "http&#58;&#47;&#47;www&#46;imf&#46;org&#47;external&#47;datamapper&#47;fiscalrules&#47;map&#47;map&#46;htm" on this server.<P>
Reference&#32;&#35;18&#46;c3aa3717&#46;1781288360&#46;db11f77
<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;18&#46;c3aa3717&#46;1781288360&#46;db11f77</P>
</BODY>
</HTML>



In [13]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
test_urls = {
    'EPI results CSV 2024': 'https://epi.yale.edu/downloads/2024epiresults.csv',
    'EPI results ZIP 2024': 'https://epi.yale.edu/downloads/2024epiresultsvXX.zip',
    'WB Carbon Aug 2025': 'https://carbonpricingdashboard.worldbank.org/sites/default/files/carbon-pricing-dashboard-data/data_08_2025.xlsx',
    'WB Carbon Jun 2026': 'https://carbonpricingdashboard.worldbank.org/sites/default/files/carbon-pricing-dashboard-data/data_06_2026.xlsx',
}

for name, url in test_urls.items():
    try:
        r = requests.head(url, timeout=10, allow_redirects=True)
        ct = r.headers.get('Content-Type', '')[:40]
        print(f"{name}: {r.status_code} [{ct}]")
    except Exception as e:
        print(f"{name}: ERROR — {e}")

EPI results CSV 2024: 404 [text/html; charset=utf-8]
EPI results ZIP 2024: 404 [text/html; charset=utf-8]
WB Carbon Aug 2025: 403 [text/html; charset=UTF-8]
WB Carbon Jun 2026: 403 [text/html; charset=UTF-8]


In [14]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Scrape EPI downloads page for actual CSV URLs
epi_page = requests.get('https://epi.yale.edu/downloads', headers=BROWSER_HEADERS, timeout=30)
print(f"EPI page status: {epi_page.status_code}")
csv_links = re.findall(r'https?://[^\s"\'<>]+\.(?:csv|zip)', epi_page.text)
print(f"EPI CSV/ZIP links: {csv_links[:10]}")

# Scrape WB Carbon page for actual Excel URL
carbon_page = requests.get('https://carbonpricingdashboard.worldbank.org/resources', headers=BROWSER_HEADERS, timeout=30)
print(f"\nWB Carbon page status: {carbon_page.status_code}")
xlsx_links = re.findall(r'https?://[^\s"\'<>]+\.xlsx', carbon_page.text)
print(f"WB Carbon XLSX links: {xlsx_links[:5]}")

EPI page status: 200
EPI CSV/ZIP links: ['https://epi.yale.edu/downloads/epi2024raw.zip', 'https://epi.yale.edu/downloads/epi2024indicators.zip', 'https://epi.yale.edu/downloads/epi2024results.csv', 'https://epi.yale.edu/downloads/epi2024variables2024-12-11.csv', 'https://epi.yale.edu/downloads/epi2024weights.csv', 'https://epi.yale.edu/downloads/epi2024targets.csv']

WB Carbon page status: 403
WB Carbon XLSX links: []


In [15]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Test EPI results CSV
epi_url = 'https://epi.yale.edu/downloads/epi2024results.csv'
r = requests.get(epi_url, headers=BROWSER_HEADERS, timeout=30)
print(f"EPI results: {r.status_code}, size={len(r.content)/1024:.1f}KB")
if r.status_code == 200:
    import io
    df = pd.read_csv(io.StringIO(r.text))
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns[:10])}")

# Try WB Carbon with different URL patterns
carbon_urls = [
    'https://carbonpricingdashboard.worldbank.org/sites/default/files/2025-08/data_08_2025.xlsx',
    'https://carbonpricingdashboard.worldbank.org/sites/default/files/2026-06/data_06_2026.xlsx',
    'https://carbonpricingdashboard.worldbank.org/sites/default/files/2025-09/data_09_2025.xlsx',
]
for url in carbon_urls:
    r = requests.head(url, timeout=10, allow_redirects=True)
    print(f"\nWB Carbon {url.split('/')[-1]}: {r.status_code} [{r.headers.get('Content-Type','')[:30]}]")

EPI results: 200, size=120.4KB
Shape: (180, 149)
Columns: ['code', 'iso', 'country', 'EPI.old', 'EPI.new', 'ECO.old', 'ECO.new', 'BDH.old', 'BDH.new', 'MKP.old']

WB Carbon data_08_2025.xlsx: 403 [text/html; charset=UTF-8]

WB Carbon data_06_2026.xlsx: 403 [text/html; charset=UTF-8]

WB Carbon data_09_2025.xlsx: 403 [text/html; charset=UTF-8]


In [16]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Test World Carbon Pricing Database on GitHub (RFF/Dolphin)
carbon_github_urls = [
    'https://raw.githubusercontent.com/g-dolphin/WorldCarbonPricingDatabase/master/data/WorldCarbonPricingDatabase_nationalLevel.csv',
    'https://raw.githubusercontent.com/g-dolphin/WorldCarbonPricingDatabase/master/data/WorldCarbonPricingDatabase.csv',
    'https://api.github.com/repos/g-dolphin/WorldCarbonPricingDatabase/contents/data',
]

for url in carbon_github_urls:
    try:
        r = requests.head(url, timeout=10, allow_redirects=True)
        ct = r.headers.get('Content-Type', '')[:40]
        size = int(r.headers.get('Content-Length', 0))/1024
        print(f"{url.split('/')[-1][:40]}: {r.status_code} [{ct}] {size:.0f}KB")
    except Exception as e:
        print(f"ERROR: {e}")

WorldCarbonPricingDatabase_nationalLevel: 404 [text/plain; charset=utf-8] 0KB
WorldCarbonPricingDatabase.csv: 404 [text/plain; charset=utf-8] 0KB
data: 404 [application/json; charset=utf-8] 0KB


In [17]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Test OWID carbon pricing CSV
owid_carbon_urls = [
    'https://ourworldindata.org/grapher/carbon-tax-instruments.csv?v=1&csvType=full&useColumnShortNames=false',
    'https://ourworldindata.org/grapher/carbon-prices.csv?v=1&csvType=full&useColumnShortNames=false',
]

for url in owid_carbon_urls:
    r = requests.get(url, timeout=30)
    print(f"{url.split('/')[-1].split('.')[0]}: {r.status_code}, size={len(r.content)/1024:.1f}KB")
    if r.status_code == 200 and 'text/csv' in r.headers.get('Content-Type', ''):
        df = pd.read_csv(io.StringIO(r.text))
        print(f"  Shape: {df.shape}, columns: {list(df.columns[:5])}")

# Test GitHub WCPD data files
github_urls = [
    'https://raw.githubusercontent.com/g-dolphin/WorldCarbonPricingDatabase/master/_dataset/data/WorldCarbonPricingDatabase_national.csv',
    'https://raw.githubusercontent.com/g-dolphin/ECP/master/ecp_national.csv',
]
for url in github_urls:
    r = requests.head(url, timeout=10)
    print(f"\n{url.split('/')[-1]}: {r.status_code} [{r.headers.get('Content-Type','')[:30]}]")

carbon-tax-instruments: 200, size=238.2KB
  Shape: (7437, 4), columns: ['Entity', 'Code', 'Year', 'Covered by tax instrument in at least one sector']
carbon-prices: 404, size=0.0KB

WorldCarbonPricingDatabase_national.csv: 404 [text/plain; charset=utf-8]

ecp_national.csv: 404 [text/plain; charset=utf-8]


In [19]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Try OECD newer API formats
oecd_urls2 = [
    'https://sdmx.oecd.org/public/rest/data/OECD.TAD.TC,DF_TFI/all?format=csvfilewithlabels',
    'https://sdmx.oecd.org/public/rest/data/OECD,TFI/all?format=csv',
    'https://sdmx.oecd.org/public/rest/dataflow/OECD.TAD.TC/DF_TFI',
    'https://sdmx.oecd.org/public/rest/data/OECD.TAD.TC,DF_TFI,1.0/all?format=csv',
]

for url in oecd_urls2:
    try:
        r = requests.head(url, timeout=10, allow_redirects=True)
        ct = r.headers.get('Content-Type', '')[:40]
        size = int(r.headers.get('Content-Length', 0))/1024
        print(f"{r.status_code}: {url.split('/')[-1][:50]} [{ct}] {size:.0f}KB")
    except Exception as e:
        print(f"ERROR: {e}")

404: all?format=csvfilewithlabels [text/plain] 0KB
404: all?format=csv [text/plain] 0KB
404: DF_TFI [text/plain] 0KB
404: all?format=csv [text/plain] 0KB


In [26]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Inspect CIVICUS countries API
r = requests.get('https://monitor.civicus.org/api/countries/', headers=BROWSER_HEADERS, timeout=30)
print(f"Status: {r.status_code}, Size: {len(r.content)/1024:.1f}KB")
data = r.json()
print(f"Type: {type(data)}")
if isinstance(data, list):
    print(f"Length: {len(data)}")
    print(f"First item keys: {list(data[0].keys()) if data else 'empty'}")
    print(f"First item: {data[0]}")
elif isinstance(data, dict):
    print(f"Keys: {list(data.keys())}")
    print(f"First 500 chars: {str(data)[:500]}")

Status: 200, Size: 251.9KB
Type: <class 'list'>
Length: 199
First item keys: ['name', 'status', 'current_score', 'ratings']
First item: {'name': 'Afghanistan', 'status': 'Closed', 'current_score': 7.68, 'ratings': [{'score': '7.68', 'created_at': '2025-12-06T14:25:19.613598Z', 'rating': 'Closed'}, {'score': '16.34', 'created_at': '2025-12-06T14:21:21.372429Z', 'rating': 'Closed'}, {'score': '11.00', 'created_at': '2024-12-03T20:14:29.748519Z', 'rating': 'Closed'}, {'score': '11.04', 'created_at': '2024-12-03T10:34:32.752241Z', 'rating': 'Closed'}, {'score': '19.50', 'created_at': '2024-12-03T10:06:20.360684Z', 'rating': 'Closed'}, {'score': '12.16', 'created_at': '2023-12-05T15:33:46.693374Z', 'rating': 'Closed'}, {'score': '26.24', 'created_at': '2023-12-05T09:33:20.671373Z', 'rating': 'Repressed'}, {'score': '13.43', 'created_at': '2023-11-20T18:58:03.395259Z', 'rating': 'Closed'}, {'score': '40.27', 'created_at': '2023-11-20T18:51:50.388335Z', 'rating': 'Repressed'}, {'score': '26.2